# Spark feature job on HI-Large (final run)

Runs only after `test_spark.ipynb` has written **PASS** to S3. The job reads the 17 GB HI-Large CSV, writes the features partitioned by day, and writes the manifest. `train_triage_xgboost.ipynb` then trains on those features.

Before you run it: upload `HI-Large_Trans.csv` to `s3://neu-datasets/`.

## 1. Configuration

In [ ]:
from pathlib import Path
import shutil

import boto3
import sagemaker

ROLE_ARN = "arn:aws:iam::157205473853:role/AmazonSageMaker-ExecutionRole-20261020T232149"  # the notebook's execution role
REGION = "ap-south-1"
BUCKET = "neu-datasets"
NB_DIR = Path.cwd()  # run from the notebooks/ folder
WORK = NB_DIR / "sagemaker_work"
JOB = WORK / "job"  # everything the SageMaker jobs need, bundled here
sess = sagemaker.Session(boto_session=boto3.Session(region_name=REGION))
s3 = boto3.client("s3", region_name=REGION)


def build_job_folder():
    """Copies the Spark job, the rule module, the parity check and triage_lib into one folder for SageMaker."""
    shutil.rmtree(JOB, ignore_errors=True)
    JOB.mkdir(parents=True)
    for f in (NB_DIR / "spark").glob("*.py"):
        shutil.copy(f, JOB)
    shutil.copytree(NB_DIR / "triage_lib", JOB / "triage_lib", ignore=shutil.ignore_patterns("__pycache__"))
    (JOB / "requirements.txt").write_text("pandas==3.0.6\nnumpy==2.5.4\npyarrow==26.0.0\n", encoding="utf-8")
    print("job folder:", sorted(p.name for p in JOB.iterdir()))

HORIZON = 7                          # label horizon in days
PARTITIONS = 800                     # shuffle partitions for the full file
CSV_KEY = "HI-Large_Trans.csv"
VERDICT_KEY = "parity/sample/verdict/verdict.txt"
FEATURES_PREFIX = "features/hi_large"

## 2. Gate: the sample parity verdict must be PASS

In [ ]:
verdict = s3.get_object(Bucket=BUCKET, Key=VERDICT_KEY)["Body"].read().decode("utf-8")
print(verdict)
if not verdict.startswith("PASS"):
    raise SystemExit("Parity is not OK. Run test_spark.ipynb until it passes, then come back here.")
print("gate passed")

## 3. Bundle the jobs

In [ ]:
build_job_folder()

## 4. Run the Spark job on HI-Large

Four `ml.r5.4xlarge` instances, 300 GB volume each. The per-account rule pass is the slowest stage. Runtime is hours, not minutes.

In [ ]:
from sagemaker.spark.processing import PySparkProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput

spark = PySparkProcessor(
    base_job_name="triage-spark-hi-large", framework_version="3.5", role=ROLE_ARN,
    instance_count=4, instance_type="ml.r5.4xlarge", volume_size_in_gb=300,
    max_runtime_in_seconds=6 * 3600, sagemaker_session=sess,
)
spark.run(
    submit_app=str(JOB / "build_features_spark.py"),
    submit_py_files=[str(JOB / "rules_r1_r4.py")],
    arguments=[
        "--input", f"/opt/ml/processing/input/csv/{CSV_KEY}",
        "--output", "/opt/ml/processing/output/features",
        "--manifest", "/opt/ml/processing/output/manifest/features_manifest.json",
        "--horizon", str(HORIZON), "--partitions", str(PARTITIONS),
    ],
    inputs=[ProcessingInput(source=f"s3://{BUCKET}/{CSV_KEY}", destination="/opt/ml/processing/input/csv")],
    outputs=[
        ProcessingOutput(source="/opt/ml/processing/output/features", destination=f"s3://{BUCKET}/{FEATURES_PREFIX}/features"),
        ProcessingOutput(source="/opt/ml/processing/output/manifest", destination=f"s3://{BUCKET}/{FEATURES_PREFIX}/manifest"),
    ],
)
print("HI-Large feature job finished")

## 5. Check the manifest

The rows and positives must be plausible for HI-Large (roughly 97 days, about 223k laundering transactions in the source file).

In [ ]:
import json

manifest = json.loads(s3.get_object(Bucket=BUCKET, Key=f"{FEATURES_PREFIX}/manifest/features_manifest.json")["Body"].read())
print({k: manifest[k] for k in ("day0", "max_day", "horizon", "rows", "positives", "engine")})
assert manifest["horizon"] == HORIZON
print("next: open train_triage_xgboost.ipynb")